<h1 style="font-size: 36px; color: blue;">STOWA Proevenverzameling tool v5.0</h1>

Deze Jupyter notebook bevat functies opgenomen in de python package PV-tooling ### voor het opstellen van lokale of regionale proevenverzamelingen voor het bepalen van geotechnische parameters. De methode is ontwikkeld voor het uitvoeren van analyses in relatie tot de geotechnische stabiliteit van dijken, maar kan ook breder worden toegepast. De notebook dient tevens als handleiding om de gebruiker stapsgewijs te ondersteunen bij het opstellen van een proevenverzameling.

Met de beschikbare functies kunnen zowel gedraineerde als ongedraineerde sterkteparameters worden berekend alsmede enkele samendrukkingsparameters. Van deze parameters worden verwachtingswaarde, karakteristieke waarde en rekenwaarde bepaald. 

De functies zijn opgesteld conform de werkwijze beschreven in [Statistische methoden t.b.v. proevenverzamelingen, DIV, v1.0], zie tevens: https://xxxxxxxxxxxxxxxxx. De tool bevat tevens hulpmiddelen voor het onderscheiden of samenvoegen van groepen in een verzameling op basis van verschillende kenmerken. 

De onderliggende data om een proevenverzameling samen te stellen is beschreven in een vaste structuur. Deze structuur is vastgelegd in een uitwisselformat. Het uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx. De geotechnische laboratoria kennen deze database en kunnen deze database vullen met resultaten van grond- en laboratoriumonderzoek. Op deze wijze ontstaat er uniformering op het gebied van data-uitwisseling en –opslag van proefresultaten.

NB. De verantwoordelijkheid voor het gebruik van deze tools ligt bij de gebruiker.


In [1]:
# Importeren van benodigde packages

import importlib.util

# Functie om te controleren of een package is geïnstalleerd
def check_package_install(package_name):
    package_spec = importlib.util.find_spec(package_name)
    if package_spec is None:
        print(f"{package_name} is niet geïnstalleerd. Installatie wordt gestart...")
        !pip install {package_name}
    else:
        print(f"{package_name} is al geïnstalleerd.")

# Controleer en installeer openpyxl indien nodig
check_package_install("openpyxl")

# Controleer en installeer ipyfilechooser indien nodig
check_package_install("ipyfilechooser")


# Controleer en installeer ipyfilechooser indien nodig
#check_package_install("jupyter_contrib_nbextensions")


# Installeer Jupyter Notebook Extensions
#!pip install --upgrade pip
#!pip install notebook jupyter jupyter_contrib_nbextensions
#!pip install jupyter_nbextensions_configurator

# Installeer de extensies
#!jupyter contrib nbextension install --user

# Schakel een specifieke extensie in
#!jupyter nbextension enable varInspector/main


openpyxl is al geïnstalleerd.
ipyfilechooser is al geïnstalleerd.


## Stap 1: Inladen van benodigde data

De benodigde data voor het bepalen van geotechnische parameters kan op 3 manieren worden ingeladen in deze notebook.
- Op basis van de data in de excel versie van de proevenverzamelingtool (vanaf versie 4.2n of hoger) waarin zowel reeds proevenverzamelingen zijn onderscheiden als geotechnische parameters zijn vastgesteld. NB. de onderscheiden verzamelingen worden als uitgangspunt worden ingeladen, de sterkteparameters worden opnieuw berekend. Onderliggende keuzes zoals het gewenste rekpercentage en de gekozen raaklijnen worden niet ingelezen uit de Excelversie. 
- Op basis van het Excel Uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx waarin nog geen verzamelingen onderscheiden zijn.

Zie tevens: https://github.com/kkpdata/Proevenverzamelingentool/

De proevenverzamelingtool of het uitwisselformat worden omgezet naar het template proevenverzamelingtool 5.0 

- Op basis van het Excel template voor de Proevenverzameling 5.0. Deze bestaat uit de data conform het uitwisselformat 4.2 aangevuld met de benodigde in en uitvoerdata van voorliggende tooling.


In [15]:
#Laden van widgets voor uploaden bestanden

import ipywidgets as widgets
from IPython.display import display, Markdown

# Maak de upload-widget aan
upload_widget = widgets.FileUpload(
    accept='xlsm',  # Laat alleen xlsm bestanden toe
    multiple=False  # Sta slechts één bestand per keer toe
    )

# Toon Markdown vanuit een codecel
display(Markdown("**Laad bestaande proevenverzamelingtool 4.2n of hoger:**"))

# Toon de upload-widget
display(upload_widget)

# Toon Markdown vanuit een codecel
display(Markdown("**Laad STOWA uitwisselingsformat 4.2x:**"))

# Toon de upload-widget
display(upload_widget)

# Toon Markdown vanuit een codecel
display(Markdown("**Laad Template proevenverzamelingtool 5.0:**"))

# Toon de upload-widget
display(upload_widget)

print()

display(Markdown("**Stap 2+3: Validatie en aanvullen data overslaan (indien reeds een gevalideerde template proevenverzamelingtool 5.0 beschikbaar is):**"))

# Maak een checkbox
checkbox = widgets.Checkbox(
    value=False, 
    description='Aan/Uit', 
    disabled=False,
    indent=False  # Zorg ervoor dat er geen extra inspringing is
)

# Pas de layout aan met margin en padding
checkbox.layout = widgets.Layout(
    margin='0px 0px 0px 0px',  # Geen marge toevoegen
    width='auto'  # Past zich flexibel aan
)

# Toon de checkbox
display(checkbox)

print()


**Laad bestaande proevenverzamelingtool 4.2n of hoger:**

FileUpload(value=(), accept='xlsm', description='Upload')

**Laad STOWA uitwisselingsformat 4.2x:**

FileUpload(value=(), accept='xlsm', description='Upload')

**Laad Template proevenverzamelingtool 5.0:**

FileUpload(value=(), accept='xlsm', description='Upload')

**Stap 2: Validatie data overslaan (indien reeds een gevalideerde template proevenverzamelingtool 5.0 beschikbaar is):**

Checkbox(value=False, description='Aan/Uit', indent=False, layout=Layout(margin='0px 0px 0px 0px', width='auto…

## Stap 2: Valideren data en aanvullen benodigde imput voor afleiden parameters


De validatie bestaat uit de volgende controles: voor de verschillende typen proeven – classificatie, CRS-proef, samendrukkingsproef, DSS-proef en triaxiaalproef – wordt per regel gecontroleerd of er een proef is uitgevoerd (data ingevuld) en zo ja, of de velden die in de PV-tool benodigd zijn volledig en correct zijn ingevuld.

Het resultaat van de validatie wordt opgeslagen in een Excel-bestand, waarin per gevalideerde kolom en regel het resultaat wordt weergegeven. Daarnaast wordt per kolom aangegeven hoeveel fouten zijn aangetroffen. Ook wordt vermeld of een bepaalde kolom noodzakelijk is voor de functionaliteit van de tooling of dat de tooling ook zonder die kolom kan worden uitgevoerd.



In [23]:
#Run validatie


#Geef terug of er fouten zijn waardoor functies (deels) niet werken. Op deze punten moet de geimporteerde data worden aangepast of aangevuld.
#en zo ja het aantal fouten per categorie
#Even kijken wat handig is, we zouden het aantal fouten terug kunnen geven per categorie, dus: boringen, classificatie, CSR, samenddrukking, DSS en Triaxiaal

#geef terug of er fouten zijn waarvan wordt aanbevolen om deze aan te passen of aan te vullen omdat het ontbreken hiervan het lastigter maakt om verzamelingen correct te onderscheiden. 
#en zo ja het aantal fouten per categorie


In [24]:
#Functie validatie naar excel te exporteren

import ipywidgets as widgets
from IPython.display import display, Markdown
import pandas as pd
from ipyfilechooser import FileChooser
import os

# Output-widget voor uitvoer tonen in notebook
output = widgets.Output()

# Maak een FileChooser aan die alleen mappen toont (start in huidige werkmap)
fc = FileChooser('.')
fc.show_only_dirs = True

# Maak de export-knop aan
button = widgets.Button(description="Exporteer")

def on_button_clicked(b):
    with output:
        output.clear_output()  # Wis vorige uitvoer
        
        # Controleer of er een geldige map is geselecteerd
        gekozen_map = fc.selected_path
        if not gekozen_map or not os.path.isdir(gekozen_map):
            print("Selecteer eerst een geldige map.")
            return
        
        # Dummy-data maken als DataFrame
        data = {
            "Monster": ["A", "B", "C"],
            "Proef": ["TXT", "DSS", "TXT"],
            "Sterkte": [20, 15, ""],
            "Resultaat validatie": ["OK", "OK", "Error"]
        }
        df = pd.DataFrame(data)
        
        # Bestandspad samenstellen: gekozen map + bestandsnaam
        bestand_naam = f"{gekozen_map}/validatie_data.xlsx"
        
        # Exporteer DataFrame naar Excel zonder indexkolom
        df.to_excel(bestand_naam, index=False)
        
        print(f"Excel-bestand '{bestand_naam}' succesvol geëxporteerd!")
        
# Koppel functie aan knopklik    
button.on_click(on_button_clicked)

# Toon instructie-tekst en widgets    
display(Markdown("**Kies een map om de Excel-bestanden op te slaan en exporteer de resultaten van de validatie.**"))
display(fc)       # Mapkiezer tonen  
display(button)   # Knop tonen  
display(output)   # Output-widget tonen




**Kies een map om de Excel-bestanden op te slaan en exporteer de resultaten van de validatie.**

FileChooser(path='C:\python_cursus\PVtool2025_github', filename='', title='', show_hidden=False, select_desc='…

Button(description='Exporteer', style=ButtonStyle())

Output()

## Stap 3: Aanvulllen benodigde input voor afleiden parameters

Aanvullend op de proefdata is voor het afleiden van de ongedraineerde schuifsterkteparameters specifieke informatie benodigd bij de beschikbare DSS- en triaxiaalproeven in de PV-tool. Deze data wordt automatisch toegevoegd aan het template voor de proevenverzamelingtool 5.0 indien de validatie goed doorlopen is, maar moet door de gebruiker op een aantal punten nog wel worden gecontroleerd. 

- Betreft het een OC of NC proef, oftewel is de proef geconsolideerd bij de geschatte dagelijkse effectieve terreinspanning (OCR>1) of (ruim) boven de grensspanning (OCR=1). De tooling doet hier een voorstel voor door de consolidatiespanning (sigma'vc) te vergelijken met de terreinspanning (sigma'vi). Indien deze niet meer dan +30% afwijkt wordt aangenomen dat het een OC-proef betreft en bij grotere afwijkingen een NC-proef. Het is noodzakelijk om dit voorstel te controleren. In de regel is een OC-proef vrijwel altijd dicht bij de terreinspanning gekozen en ligt een NC-proef er ruim boven, maar hier zijn altijd uitzonderingen op mogelijk. Deze data wordt weggeschreven in de velden (ANA_TXT_CONSOLIDATIE_TYPE_VOORSTEL) en ANA_DSS_CONSOLIDATIE_TYPE_VOORSTEL). Het voorstel kan handmatig overschreven worden in de kolom (ANA_TXT_CONSOLIDATIE_TYPE_HANDMATIG) en (ANA_DSS_CONSOLIDATIE_TYPE_HANDMATIG)
  
- Betreft het een OC-proef dan is er een waarde benodigd van de overconsolidatieratio, oftwel OCR van het monster bij het afleiden van de Shansep parameters op basis van de methode "Lineaire regressie: bepaling S en m uit triaxiaal- of DSS proeven". OCR = grensspanning (sigma'vy) / vertikale consolidatiespanning (sigma'vc). De PV-tool vereist hiervoor per monster een schatting van schatting van de grensspanning. Indien er bij dezelfde bus een resultaat beschikbaar is uit een samendrukkingsproef of CRS-proef wordt deze gebruikt. Indien deze ontbreekt wordt een voorstel gedaan op basis van de gemiddelde POP (POP = grensspanning - effectieve terreinspanning o.b.v. de monsters waar wel een proef is uitgevoerd ) per boring. Indien er bij de betreffende boring geen grensspanning beschikbaar is wordt waarde Error ingevuld. Het is dan niet mogelijk om geautomatiseerd een OCR te schatten. Deze data wordt weggeschreven in de velden (ANA_GRENSSPANNING_VOORSTEL). Het is noodzakelijk om de voorgestelde waarden te controleren. Als gebruiker kan je het voorstel overrulen in de kolom (ANA_GRENSSPANNING_HANDMATIG).

NB. Bij gebruik van de bestaande Excel versie van de proevenverzamelingtool zijn de cellen ANA_GRENSSPANNING_HANDMATIG veelal reeds ingevuld. In dat geval worden deze waarden overgenomen. 

- de verzameling waartoe een uitgevoerde proef behoort wordt overgenomen wanneer een bestaande excel proevenverzamelingtool wordt ingeladen en weggeschreven in het veld (PV_NAAM). Als er geen waarde beschikbaar is wordt de naam "TXT-proef" of "DSS-proef" ingevuld. De gebruiker kan hier naar eigen inzicht grondlagen in onderscheiden. Dit kan door dit handmatig op te geven in het template of met behulp van de functies om groepen te onderscheiden. Per onderscheiden verzameling kunnen in stap 4 parameters worden afgeleid. 


In [ ]:
#run aanvullen benodigde input.

#ALG__BORING_MONSTERNR_ID berekenen conform functie die reeds door Tjalda is gemaakt.

#aanvullend op de data worden de volgende kolommen toegevoegd
#als ALG__DSS = true 
#of als ALG__TRIAXIAAL = true 
#dan gaan we deze cellen ook vullen.

#ANA_TERREINSPANNING	(conform ANA functie die reeds door Tjalda is gemaakt)
#ANA_TXT_CONSOLIDATIE_TYPE_VOORSTEL, als ANA_DSS_MAX_CONSOLIDATIE_SPANNING niet meer dan +30% afwijkt van [ANA_TERREINSPANNING] dan OC, anders NC
#ANA_TXT_CONSOLIDATIE_TYPE_HANDMATIG, leeg laten, gebruiker kan hier in template automatisch bepaalde waarde overrulen.
#ANA_DSS_CONSOLIDATIE_TYPE_VOORSTEL als ANA_TXT_MAX_CONSOLIDATIE_SPANNING niet meer dan +30% afwijkt van [ANA_TERREINSPANNING] dan OC, anders NC
#ANA_DSS_CONSOLIDATIE_TYPE_HANDMATIG, leeg laten, gebruiker kan hier in template automatisch bepaalde waarde overrulen.


#ANA_GRENSSPANNING_PROEF (conform ANA_GRENSSPANNING die reeds door Tjalda is gemaakt)

#volgende alleen voor OC proeven!
#ANA_GRENSSPANNING_VOORSTEL (deze is nieuw, op basis van de functies reeds door Tjalda gemaakt ANA_POP_VELD	en ANA_POP_VELD_GEMIDDELD 
#doen we een voorstel voor de grensspanning als een proefwaarde ontbreekt. 
#Als er bij de betreffende boring geen grensspanning beschikbaar is wordt waarde Error ingevuld.

#ANA_GRENSSPANNING_HANDMATIG (bij het inladen van de bestaande PVtool wordt deze waarde overgenomen, bij het uitwisselformat is deze leeg en mag de gebruiker deze invullen) 
#ANA_GRENSSPANNING_REKEN (deze pakt de te hanteren waarde, handmatig is leidend, dan voorstel of proef, als er wel een proef is en geen grensspanning dan error weergeven)

#ANA_OCR_TXT_MONSTER deze kan berekend worden door de grensspanning te delen door ANA_TERREINSPANNING bij de OC proeven, bij NC proeven is waarde 1.
#ANA_OCR_DSS_MONSTER deze kan berekend worden door de grensspanning te delen door ANA_TERREINSPANNING bij de OC proeven, bij NC proeven is waarde 1.


#PV_NAAM (bij het inladen van de bestaande PVtool wordt deze waarde overgenomen, anders bij TXT proef default "TXT-proef" invullen en "DSS-proef" bij DSS.
#PV_OPMERKING (bij het inladen van de bestaande PVtool wordt deze waarde overgenomen, anders leeg)

#deze kolommen vormen samen met de data het template voor de PV tool. hier een dataframe van maken voor export en om verder te gaan in stap 4.

In [22]:
#Functie om template naar excel te exporteren

# Koppel functie aan knopklik    
button.on_click(on_button_clicked)

# Toon instructie-tekst en widgets    
display(Markdown("**Kies een map om de Excel-bestanden op te slaan en exporteer het template PV-tool 5.0.**"))
display(fc)       # Mapkiezer tonen  
display(button)   # Knop tonen  
display(output)   # Output-widget tonen



**Kies een map om de Excel-bestanden op te slaan en exporteer het template PV-tool 5.0.**

FileChooser(path='C:\python_cursus\PVtool2025_github', filename='', title='', show_hidden=False, select_desc='…

Button(description='Exporteer', style=ButtonStyle())

Output()

#### Stap 4: Bepalen parameters




